# Claim-Frequency model — A/B arm: `OrdinalSelector`

Copy of `frequency_model_2026.ipynb` with **one variable changed**: feature selection runs
through `OrdinalSelector` instead of `ClassificationSelector`. Everything else — data
prep, split, carving config, `n_best_features=100`, redundancy filters, `N_TRIALS=300`
and the Optuna seed — is identical, so the difference in dev log loss is attributable.

**Result: this arm is worse — 0.95366 dev log loss**, against the shipped arm's
**0.91991** and the 2025 winner's **0.91936**. It ships as the negative result it is.

Two cautions on how far that may be read. The gap sits ~8 standard deviations above the
seed spread measured by `tools/seed_variance.py`, so it is probably not noise — but it is
one draw on this feature set compared against a four-draw distribution on another, which
is not a rigorous comparison. And it is unexplained: `tools/cmp_selectors.py` and
`tools/measure_grid.py` show the two selectors agreeing at 0.96 rank correlation with 93
of 100 features in common. Supported: *did not improve the frequency arm*. Not supported:
any figure for what it costs.

**Export cells are removed** so this arm cannot overwrite the measured hand-off files
(`data/frequency_2026.csv`, `data/oos_frequency_2026.csv`) or the severity target carver.


## Loading data & target  *(reused from 2025)*

In [1]:
import pandas as pd

data_path = "../data/"

# loading x_train
data = pd.read_csv(data_path + "train_input_Z61KlZo.csv", low_memory=False)
data.set_index("ID", inplace=True)
print("x_train", data.shape)

# loading target
target = pd.read_csv(data_path + "train_output_DzPxaPY.csv", low_memory=False)
target.set_index("ID", inplace=True)
print("y_train", target.shape)

# joining x_train and y_train
data = data.join(target.drop("ANNEE_ASSURANCE", axis=1))
print("data", data.shape)

x_train (383610, 373)
y_train (383610, 4)


data (383610, 376)


In [2]:
target_col = "TARGET"
data[target_col] = (data["FREQ"] * data["ANNEE_ASSURANCE"]).astype(int)
data[target_col].value_counts(normalize=False).sort_index()

TARGET
0    381061
1      2458
2        87
3         2
4         1
5         1
Name: count, dtype: int64

## Stratified sampling & inverse-frequency weights  *(reused from 2025)*

In [3]:
import numpy as np

from collections import Counter
from sklearn.model_selection import train_test_split
from data_toolkit import collapse_count

# Compute class frequencies
class_counts = Counter(collapse_count(data[target_col]))
total_samples = len(data[target_col])

# Compute inverse frequency class weights
class_weights = {
    cls: total_samples / (len(class_counts) * count)
    for cls, count in class_counts.items()
}
print("Class weights:", class_weights)

# Assign sample weights based on target values
weights = np.array([class_weights[label] for label in collapse_count(data[target_col])])

# Train-test split
x_train, x_dev, y_train, y_dev, w_train, w_dev = train_test_split(
    data,
    data[target_col],
    weights,
    test_size=0.2,
    random_state=42,
    stratify=collapse_count(data[target_col]),
)
print("y_train mean", y_train.mean(), " observations:", x_train.shape[0])
print("y_dev mean", y_dev.mean(), " observations:", x_dev.shape[0])

Class weights: {0: 0.3355630725789309, 1: 52.0219690805533, 2: 1405.1648351648353}


y_train mean 0.006895023591668622  observations: 306888
y_dev mean 0.006921091733792133  observations: 76722


## Feature engineering  *(reused from 2025)*

In [4]:
from data_toolkit import Processor

proc = Processor()
x_train = proc.fit_transform(x_train)
x_dev = proc.transform(x_dev)
data = proc.transform(data)

## Column typing  *(reused from 2025)*

In [5]:
from AutoCarver import Features

# sorting columns per type
features = Features.from_dataframe(x_train)

# getting ordinal columns
ordinals = [
    "NB_CASERNES",
    "BDTOPO_BAT_MAX_HAUTEUR",
    "HAUTEUR_MAX",
    "HAUTEUR",
    "BDTOPO_BAT_MAX_HAUTEUR_MAX",
    "MEN_SURF",
    "IND_SNV",
    "IND_INC",
    "IND_Y9",
    "IND_0_Y1",
    "IND",
    "LOG_SOC",
    "LOG_INC",
    "LOG_APA3",
    "LOG_AVA1",
    "MEN_MAIS",
    "MEN_COLL",
    "MEN_FMP",
    "MEN_PROP",
    "MEN_PAUV",
    "MEN",
    "COEFASS",
]
ordinals += [
    "DISTANCE_111",
    "DISTANCE_112",
    "DISTANCE_121",
    "DISTANCE_122",
    "DISTANCE_123",
    "DISTANCE_124",
    "DISTANCE_131",
    "DISTANCE_132",
    "DISTANCE_133",
    "DISTANCE_141",
    "DISTANCE_142",
    "DISTANCE_211",
    "DISTANCE_212",
    "DISTANCE_213",
    "DISTANCE_221",
    "DISTANCE_222",
    "DISTANCE_223",
    "DISTANCE_231",
    "DISTANCE_242",
    "DISTANCE_243",
    "DISTANCE_244",
    "DISTANCE_311",
    "DISTANCE_312",
    "DISTANCE_313",
    "DISTANCE_321",
    "DISTANCE_322",
    "DISTANCE_323",
    "DISTANCE_324",
    "DISTANCE_331",
    "DISTANCE_332",
    "DISTANCE_333",
    "DISTANCE_334",
    "DISTANCE_335",
    "DISTANCE_411",
    "DISTANCE_412",
    "DISTANCE_421",
    "DISTANCE_422",
    "DISTANCE_423",
    "DISTANCE_511",
    "DISTANCE_512",
    "DISTANCE_521",
    "DISTANCE_522",
    "DISTANCE_523",
    "PROPORTION_11",
    "PROPORTION_12",
    "PROPORTION_13",
    "PROPORTION_14",
    "PROPORTION_21",
    "PROPORTION_22",
    "PROPORTION_23",
    "PROPORTION_24",
    "PROPORTION_31",
    "PROPORTION_32",
    "PROPORTION_33",
    "PROPORTION_41",
    "PROPORTION_42",
    "PROPORTION_51",
    "PROPORTION_52",
    "MEN_1IND",
    "MEN_5IND",
    "LOG_A1_A2",
    "LOG_A2_A3",
    "IND_Y1_Y2",
    "IND_Y2_Y3",
    "IND_Y3_Y4",
    "IND_Y4_Y5",
    "IND_Y5_Y6",
    "IND_Y6_Y7",
    "IND_Y7_Y8",
    "IND_Y8_Y9",
    "DISTANCE_1",
    "DISTANCE_2",
    "ALTITUDE_1",
    "ALTITUDE_2",
    "ALTITUDE_3",
    "ALTITUDE_4",
    "ALTITUDE_5",
    "NBJTX25_MM_A",
    "NBJTX25_MMAX_A",
    "NBJTX25_MSOM_A",
    "NBJTX0_MM_A",
    "NBJTX0_MMAX_A",
    "NBJTX0_MSOM_A",
    "NBJTXI27_MM_A",
    "NBJTXI27_MMAX_A",
    "NBJTXI27_MSOM_A",
    "NBJTXS32_MM_A",
    "NBJTXS32_MMAX_A",
    "NBJTXS32_MSOM_A",
    "NBJTXI20_MM_A",
    "NBJTXI20_MMAX_A",
    "NBJTXI20_MSOM_A",
    "NBJTX30_MM_A",
    "NBJTX30_MMAX_A",
    "NBJTX30_MSOM_A",
    "NBJTX35_MM_A",
    "NBJTX35_MMAX_A",
    "NBJTX35_MSOM_A",
    "NBJTN10_MM_A",
    "NBJTN10_MMAX_A",
    "NBJTN10_MSOM_A",
    "NBJTNI10_MM_A",
    "NBJTNI10_MMAX_A",
    "NBJTNI10_MSOM_A",
    "NBJTN5_MM_A",
    "NBJTN5_MMAX_A",
    "NBJTN5_MSOM_A",
    "NBJTNS25_MM_A",
    "NBJTNS25_MMAX_A",
    "NBJTNS25_MSOM_A",
    "NBJTNI15_MM_A",
    "NBJTNI15_MMAX_A",
    "NBJTNI15_MSOM_A",
    "NBJTNI20_MM_A",
    "NBJTNI20_MMAX_A",
    "NBJTNI20_MSOM_A",
    "NBJTNS20_MM_A",
    "NBJTNS20_MMAX_A",
    "NBJTNS20_MSOM_A",
    "NBJTMS24_MM_A",
    "NBJTMS24_MMAX_A",
    "NBJTMS24_MSOM_A",
    "TAMPLIAB_VOR_MM_A",
    "TAMPLIAB_VOR_MMAX_A",
    "TAMPLIM_VOR_MM_A",
    "TAMPLIM_VOR_MMAX_A",
    "TM_VOR_MM_A",
    "TM_VOR_MMAX_A",
    "TMM_VOR_MM_A",
    "TMM_VOR_MMAX_A",
    "TMMAX_VOR_MM_A",
    "TMMAX_VOR_MMAX_A",
    "TMMIN_VOR_MM_A",
    "TMMIN_VOR_MMAX_A",
    "TN_VOR_MM_A",
    "TN_VOR_MMAX_A",
    "TNAB_VOR_MM_A",
    "TNAB_VOR_MMAX_A",
    "TNMAX_VOR_MM_A",
    "TNMAX_VOR_MMAX_A",
    "TX_VOR_MM_A",
    "TX_VOR_MMAX_A",
    "TXAB_VOR_MM_A",
    "TXAB_VOR_MMAX_A",
    "TXMIN_VOR_MM_A",
    "TXMIN_VOR_MMAX_A",
    "NBJFF10_MM_A",
    "NBJFF10_MMAX_A",
    "NBJFF10_MSOM_A",
    "NBJFF16_MM_A",
    "NBJFF16_MMAX_A",
    "NBJFF16_MSOM_A",
    "NBJFF28_MM_A",
    "NBJFF28_MMAX_A",
    "NBJFF28_MSOM_A",
    "NBJFXI3S10_MM_A",
    "NBJFXI3S10_MMAX_A",
    "NBJFXI3S10_MSOM_A",
    "NBJFXI3S16_MM_A",
    "NBJFXI3S16_MMAX_A",
    "NBJFXI3S16_MSOM_A",
    "NBJFXI3S28_MM_A",
    "NBJFXI3S28_MMAX_A",
    "NBJFXI3S28_MSOM_A",
    "NBJFXY8_MM_A",
    "NBJFXY8_MMAX_A",
    "NBJFXY8_MSOM_A",
    "NBJFXY10_MM_A",
    "NBJFXY10_MMAX_A",
    "NBJFXY10_MSOM_A",
    "NBJFXY15_MM_A",
    "NBJFXY15_MMAX_A",
    "NBJFXY15_MSOM_A",
    "FFM_VOR_MM_A",
    "FFM_VOR_MMAX_A",
    "FXI3SAB_VOR_MM_A",
    "FXI3SAB_VOR_MMAX_A",
    "FXIAB_VOR_MM_A",
    "FXIAB_VOR_MMAX_A",
    "FXYAB_VOR_MM_A",
    "FXYAB_VOR_MMAX_A",
    "FFM_VOR_COM_MM_A_Y",
    "FFM_VOR_COM_MMAX_A_Y",
    "FXI3SAB_VOR_COM_MM_A_Y",
    "FXI3SAB_VOR_COM_MMAX_A_Y",
    "NBJRR50_MM_A",
    "NBJRR50_MMAX_A",
    "NBJRR50_MSOM_A",
    "NBJRR1_MM_A",
    "NBJRR1_MMAX_A",
    "NBJRR1_MSOM_A",
    "NBJRR5_MM_A",
    "NBJRR5_MMAX_A",
    "NBJRR5_MSOM_A",
    "NBJRR10_MM_A",
    "NBJRR10_MMAX_A",
    "NBJRR10_MSOM_A",
    "NBJRR30_MM_A",
    "NBJRR30_MMAX_A",
    "NBJRR30_MSOM_A",
    "NBJRR100_MM_A",
    "NBJRR100_MMAX_A",
    "NBJRR100_MSOM_A",
    "RR_VOR_MM_A",
    "RR_VOR_MMAX_A",
    "RRAB_VOR_MM_A",
    "RRAB_VOR_MMAX_A",
]
ordinals += ["TAILLE1", "TAILLE2"]
ordinal_columns = {
    col: list(data[col].value_counts().sort_index().index)
    for col in ordinals
    if col in data.columns
}
ordinal_columns["PROPORTION_32"] += ["10. > 90"]
ordinal_columns.update(
    {
        "CARACT4": [
            "absence de surface",
            "Surface de moins d",
            "Surface entre 501",
            "Surface entre 1001",
            "Surface entre 1501",
            "Surface de plus de",
        ],
        "SURFACE4": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "SURFACE6": [
            "0",
            "500",
            "1000",
            "1500",
            "2000",
            "2500",
            "3000",
            "3500",
            "4000",
            "4500",
            "5000",
            "5500",
            "6000",
            "6500",
            "7000",
            "7000+",
        ],
        "total_surface_2023": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "total_surface_5y": [
            "Aucun feu",
            "<10ha",
            "10-20ha",
            "20-50ha",
            "50-100ha",
            "100-200ha",
            ">200ha",
        ],
        "surface_over_forest": [
            "Absence de feu",
            "<0.05",
            "0.05-0.1",
            "0.1-0.2",
            "0.5-2",
        ],
        "fire_extinction_rates": ["Aucun feu", "<50%", "50-70%", "70-85%", ">85%"],
    }
)

# columns that are to be removed (target + no values)
to_remove = target.columns.tolist() + [target_col]
to_remove += [c for c in data.columns if "MMSOM" in c]
to_remove += ["DEROG13", "DEROG14", "DEROG16"]

# removing columns
categorical_columns = [
    col.name
    for col in features.categoricals
    if col not in to_remove and col not in ordinal_columns
]
categorical_columns += ["TYPERS"]
numerical_columns = [
    col.name
    for col in features.numericals
    if col not in to_remove
    and col not in ordinal_columns
    and col not in categorical_columns
]
print(
    len(categorical_columns),
    len(numerical_columns),
    len(ordinal_columns),
    len(categorical_columns) + len(numerical_columns) + len(ordinal_columns),
)

197 120 238 555


## 2026 carving — OrdinalCarver + multiprocessing + Wilson-CI

`n_jobs` parallelises the per-feature combination search; set it to the core
count on the competition machine. `min_freq_alpha=0.05` is the 95% Wilson
interval used to test each bin's frequency.


In [6]:
import time

from AutoCarver import Features, OrdinalCarver
from AutoCarver.discretizers import ProcessingConfig

N_JOBS = 6

# the 0/1/2+ claim-count target is ordinal -> carve with Kendall's Tau-c
y_ord_train = collapse_count(y_train)
y_ord_dev = collapse_count(y_dev)

config = ProcessingConfig(
    dropna=False, copy=False, verbose=False, n_jobs=N_JOBS, min_freq_alpha=0.05
)

### Qualitative + ordinal features

In [7]:
qualitatives = Features(categoricals=categorical_columns, ordinals=ordinal_columns)

carver = OrdinalCarver(features=qualitatives, target_scale="level", config=config)

t0 = time.perf_counter()
x_train = carver.fit_transform(x_train, y_ord_train, X_dev=x_dev, y_dev=y_ord_dev)
print(
    f"[2026] qualitative carving: {time.perf_counter() - t0:.1f}s on {N_JOBS} workers"
)
carver.summary

[OrdinalCarver] Carving:   0%|          | 0/435 [00:00<?, ?feature/s]

[OrdinalCarver] dropped 38/435 feature(s) (no robust train/dev combination): LOG_VETUSTE_REGION, MEN_num, IND_num, IND_0_Y1_num, IND_Y1_Y2_num, IND_Y4_Y5_num, IND_Y6_Y7_num, IND_Y7_Y8_num, IND_INC_num, IND_0_Y1_IND, IND_Y1_Y2_IND, IND_Y2_Y3_IND, IND_Y6_Y7_IND, IND_Y7_Y8_IND, IND_INC_IND, IND_INC, IND_0_Y1, IND, LOG_INC, MEN, PROPORTION_13, PROPORTION_12, PROPORTION_14, PROPORTION_11, PROPORTION_33, PROPORTION_41, PROPORTION_51, PROPORTION_42, PROPORTION_52, IND_Y1_Y2, IND_Y2_Y3, LOG_A1_A2, IND_Y4_Y5, IND_Y6_Y7, IND_Y7_Y8, LOG_APA3_num_LOG_TOT, LOG_AVA1_num_LOG_TOT, LOG_VETUSTE


[2026] qualitative carving: 121.5s on 6 workers


content  \
feature                    tau_b    tau_c    somersd  n_mod label                                                                   
Categorical('ACTIVIT2')    0.020905 0.002217 0.003905 4.0   0                                                        [ACT3, ACT2]   
                                                            1                           [ACT8, ACT9, ACT4, ACT6, ACT7, __OTHER__]   
                                                            2                                                                ACT1   
                                                            3                                                                ACT5   
Categorical('VOCATION')    0.042972 0.006714 0.007263 2.0   0                   [VOC1, VOC7, VOC5, VOC3, VOC2, __OTHER__, VOC4...   
...                                                                                                                           ...   
Categorical('LOG_VETUSTE') NaN      NaN      NaN      NaN   57.926829268292686                                 57.926829268292686   
                                                            64.02439024390245                                   64.02439024390245   
                                                            52.083333333333336                                 52.083333333333336   
                                                            43.292682926829265                                 43.292682926829265   
                                                            69.63855421686748                                   69.63855421686748   

                                                                                target_mean_level  \
feature                    tau_b    tau_c    somersd  n_mod label                                   
Categorical('ACTIVIT2')    0.020905 0.002217 0.003905 4.0   0                            0.002030   
                                                            1                            0.005593   
                                                            2                            0.006539   
                                                            3                            0.010893   
Categorical('VOCATION')    0.042972 0.006714 0.007263 2.0   0                            0.002050   
...                                                                                           ...   
Categorical('LOG_VETUSTE') NaN      NaN      NaN      NaN   57.926829268292686           0.007920   
                                                            64.02439024390245            0.008269   
                                                            52.083333333333336           0.008765   
                                                            43.292682926829265           0.008900   
                                                            69.63855421686748            0.008993   

                                                                                frequency  \
feature                    tau_b    tau_c    somersd  n_mod label                           
Categorical('ACTIVIT2')    0.020905 0.002217 0.003905 4.0   0                    0.051364   
                                                            1                    0.035537   
                                                            2                    0.773396   
                                                            3                    0.139702   
Categorical('VOCATION')    0.042972 0.006714 0.007263 2.0   0                    0.362458   
...                                                                                   ...   
Categorical('LOG_VETUSTE') NaN      NaN      NaN      NaN   57.926829268292686   0.020571   
                                                            64.02439024390245    0.048470   
                                                            52.083333333333336   0.020076   
                                                       

### Apply carver to dev + OOS

In [8]:
x_dev = carver.transform(x_dev)

data_path = "../data/"
oos = pd.read_csv(data_path + "test_input_5qJzHrr.csv", low_memory=False)
oos = proc.transform(oos)
oos = carver.transform(oos)

No processing for continuous features

In [9]:
quantitatives = Features(numericals=numerical_columns)

## Feature selection  *(A/B arm — `OrdinalSelector`)*

`OrdinalSelector` in place of `ClassificationSelector`, and **nothing else changed**:
same carved features, same `n_best_features=100`, same redundancy filters, same seeded
300-trial search.

The two rank features differently. `ClassificationSelector` scores qualitative candidates
with Tschuprow's T, which treats `0 / 1 / 2+` as three unordered labels — a feature that
separates "0 from 1 and 2+" scores the same as one that separates "0 and 1 from 2+".
`OrdinalSelector` treats the integer-encoded target as a rank and scores with Spearman's
rho on quantitatives and a reversed Kruskal-eta-squared on qualitatives, so a feature that
moves *with* the claim count outranks one that merely splits it.

That is the selector-side version of the argument §3.2 makes about the carver: the target
is ordered, and a tool that ignores the order is leaving that information unused.


In [10]:
from AutoCarver.selectors import (
    CramervFilter,
    OrdinalSelector,
    SelectionConfig,
    SpearmanFilter,
)

config = SelectionConfig(
    qualitative_filters=[CramervFilter(threshold=0.9)],
    quantitative_filters=[SpearmanFilter(threshold=0.9)],
)
selector = OrdinalSelector(
    qualitatives + quantitatives, n_best_features=100, config=config
)
selector.fit(x_train, y_ord_train)
print("selected:", len(selector.selected_features))
selector.summary

selected: 100


,feature,Nan,Mode,measure,association,rank,filter,redundancy,redundancy_with,selected
0,Numerical('KAPITAL32'),0.000000,0.347971,Spearman,0.057794,0.0,Spearman,0.000000,itself,True
1,Numerical('KAPITAL_MAX'),0.000000,0.248687,Spearman,0.053906,1.0,Spearman,0.824113,KAPITAL32,True
2,Numerical('SURFACE10'),0.019235,0.523992,Spearman,0.051294,2.0,Spearman,0.611342,KAPITAL32,True
3,Numerical('SURFACE1'),0.000000,0.071573,Spearman,0.050736,3.0,Spearman,0.640521,KAPITAL32,True
4,Numerical('SURFACE7'),0.010479,0.559276,Spearman,0.049353,4.0,Spearman,0.621579,SURFACE10,True
...,...,...,...,...,...,...,...,...,...,...
511,Ordinal('SURFACE4'),0.000000,0.605064,KruskalEtaSquared,0.002260,NaN,Cramerv,0.999993,SURFACE6,False
512,Ordinal('total_surface_2023'),0.000000,0.587276,KruskalEtaSquared,0.000011,NaN,None,NaN,None,False
513,Ordinal('total_surface_5y'),0.000000,0.731602,KruskalEtaSquared,0.000064,NaN,None,NaN,None,False
514,Ordinal('surface_over_forest'),0.000000,0.842874,KruskalEtaSquared,0.000015,NaN,None,NaN,None,False


In [11]:
pd.set_option("display.max_rows", None)
selector.summary

,feature,Nan,Mode,measure,association,rank,filter,redundancy,redundancy_with,selected
0,Numerical('KAPITAL32'),0.000000,0.347971,Spearman,5.779430e-02,0.0,Spearman,0.000000,itself,True
1,Numerical('KAPITAL_MAX'),0.000000,0.248687,Spearman,5.390646e-02,1.0,Spearman,0.824113,KAPITAL32,True
2,Numerical('SURFACE10'),0.019235,0.523992,Spearman,5.129417e-02,2.0,Spearman,0.611342,KAPITAL32,True
3,Numerical('SURFACE1'),0.000000,0.071573,Spearman,5.073559e-02,3.0,Spearman,0.640521,KAPITAL32,True
4,Numerical('SURFACE7'),0.010479,0.559276,Spearman,4.935349e-02,4.0,Spearman,0.621579,SURFACE10,True
5,Numerical('KAPITAL21'),0.014090,0.573636,Spearman,4.465265e-02,5.0,Spearman,0.706107,KAPITAL32,True
6,Numerical('NBBAT4'),0.000000,0.121360,Spearman,4.398885e-02,6.0,Spearman,0.834693,SURFACE1,True
7,Numerical('NBSINSTRT'),0.000000,0.727373,Spearman,4.015080e-02,7.0,Spearman,0.379864,KAPITAL32,True
8,Numerical('KAPITAL23'),0.003050,0.834321,Spearman,3.762247e-02,8.0,Spearman,0.422755,SURFACE7,True
9,Numerical('EQUIPEMENT6'),0.000000,0.100607,Spearman,3.447416e-02,9.0,Spearman,0.394208,KAPITAL21,True


In [12]:
best_features = selector.selected_features.names

## XGBoost + Optuna  *(unchanged objectives from `objectives.py`)*

The model search is the same as 2025 so any dev-metric delta is attributable to
the carving step, not the tuner. Drop `N_TRIALS` while iterating.


In [13]:
import optuna

from objectives import get_multiclass_objective

N_TRIALS = 300

objective = get_multiclass_objective(
    x_train[best_features],
    y_ord_train,
    x_dev[best_features],
    y_ord_dev,
    w_train=w_train,
    w_dev=w_dev,
)
# seeded so the tuning is reproducible: the split and every XGBoost estimator
# already use random_state=42, and carving is deterministic, so this is the last
# source of run-to-run variance.
study = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=N_TRIALS)
study.best_params

[I 2026-09-01 09:11:17,187] A new study created in memory with name: no-name-146e93e7-4fc9-48cd-ab48-2f5601581d21


<repo>/.venv\Lib\site-packages\xgboost\core.py:751: UserWarning: [09:11:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[I 2026-09-01 09:11:26,392] Trial 0 finished with value: 10.641719669532819 and parameters: {'objective': 'multi:softprob', 'n_estimators': 287, 'learning_rate': 0.9507192349792751, 'max_depth': 8, 'min_child_weight': 12, 'subsample': 0.3248149123539492, 'colsample_bytree': 0.32479561626896214, 'colsample_bylevel': 0.24646688973455957, 'gamma': 8.661761457749352, 'alpha': 6.011150117432088, 'lambda': 7.080725777960454}. Best is trial 0 with value: 10.641719669532819.


[I 2026-09-01 09:11:30,938] Trial 1 finished with value: 10.42722139295433 and parameters: {'objective': 'multi:softprob', 'n_estimators': 110, 'learning_rate': 0.9699128611767781, 'max_depth': 9, 'min_child_weight': 5, 'subsample': 0.3454599737656805, 'colsample_bytree': 0.34672360788274703, 'colsample_bylevel': 0.4433937943676302, 'gamma': 5.247564316322379, 'alpha': 4.319450186421157, 'lambda': 2.9122914019804194}. Best is trial 1 with value: 10.42722139295433.


[I 2026-09-01 09:11:43,480] Trial 2 finished with value: 1.7506374451340678 and parameters: {'objective': 'multi:softprob', 'n_estimators': 406, 'learning_rate': 0.13957991126597663, 'max_depth': 3, 'min_child_weight': 8, 'subsample': 0.5648559873736287, 'colsample_bytree': 0.8281407691144109, 'colsample_bylevel': 0.3597390257266878, 'gamma': 5.142344384136116, 'alpha': 5.924145688620425, 'lambda': 0.46450412719997725}. Best is trial 2 with value: 1.7506374451340678.


...

[I 2026-09-01 09:52:14,124] Trial 296 finished with value: 0.9773215464941947 and parameters: {'objective': 'multi:softprob', 'n_estimators': 189, 'learning_rate': 0.131591055361995, 'max_depth': 1, 'min_child_weight': 3, 'subsample': 0.5704040622587087, 'colsample_bytree': 0.25561407394420027, 'colsample_bylevel': 0.7173157949560321, 'gamma': 1.6573788306657375, 'alpha': 6.147794080751044, 'lambda': 7.330401064704333}. Best is trial 258 with value: 0.9536605284485395.


[I 2026-09-01 09:52:20,802] Trial 297 finished with value: 0.9540195596434781 and parameters: {'objective': 'multi:softprob', 'n_estimators': 205, 'learning_rate': 0.10581475985784247, 'max_depth': 1, 'min_child_weight': 4, 'subsample': 0.5521548138047109, 'colsample_bytree': 0.24432121292310194, 'colsample_bylevel': 0.6629150518660362, 'gamma': 2.208219684877636, 'alpha': 6.418895006860756, 'lambda': 7.42942053316907}. Best is trial 258 with value: 0.9536605284485395.


[I 2026-09-01 09:52:29,826] Trial 298 finished with value: 2.8082027095923547 and parameters: {'objective': 'multi:softprob', 'n_estimators': 204, 'learning_rate': 0.47605798269092375, 'max_depth': 4, 'min_child_weight': 4, 'subsample': 0.9591779376853855, 'colsample_bytree': 0.25840182535180317, 'colsample_bylevel': 0.6676142448065623, 'gamma': 2.4421969355376225, 'alpha': 6.427589245980536, 'lambda': 7.597298714168295}. Best is trial 258 with value: 0.9536605284485395.


[I 2026-09-01 09:52:36,164] Trial 299 finished with value: 0.9964907351562915 and parameters: {'objective': 'multi:softprob', 'n_estimators': 201, 'learning_rate': 0.19908213617616638, 'max_depth': 1, 'min_child_weight': 4, 'subsample': 0.5501203723492567, 'colsample_bytree': 0.29669701854653785, 'colsample_bylevel': 0.6557546909605437, 'gamma': 2.2455858585829858, 'alpha': 6.357278261553955, 'lambda': 7.4975869751530535}. Best is trial 258 with value: 0.9536605284485395.


{'objective': 'multi:softprob',
 'n_estimators': 220,
 'learning_rate': 0.07498795990965833,
 'max_depth': 1,
 'min_child_weight': 5,
 'subsample': 0.6062510906395202,
 'colsample_bytree': 0.24147325980614068,
 'colsample_bylevel': 0.6842950829495661,
 'gamma': 1.8502644776605934,
 'alpha': 6.540211548573163,
 'lambda': 6.764937382394443}

In [14]:
from objectives import get_best_multiclass_model

# 7.0.5-era helper, unchanged: prints Log Loss Train / Dev (the article's dev metric)
selected_features, model = get_best_multiclass_model(
    study.best_params,
    x_train[best_features],
    y_ord_train,
    x_dev[best_features],
    y_ord_dev,
    w_train=w_train,
    w_dev=w_dev,
    train_on_full=False,
)

used features: ['VOCATION', 'KAPITAL40', 'TYPERS', 'KAPITAL41', 'KAPITAL42', 'RISK11', 'DEROG12', 'KAPITAL43', 'KAPITAL37', 'ACTIVIT2', 'DEROG4', 'ZONE', 'RISK10', 'DEROG5', 'AN_EXERC', 'total_surface_crossed', 'INDEM2', 'NBJFXI3S10_MM_A_NBJFXI3S10_MMAX_A', 'TXAB_VOR_MM_A_TXAB_VOR_MMAX_A', 'zone_vent_extinction_rate', 'NBJFXI3S16_MM_A_NBJFXI3S16_MMAX_A', 'NBJTX0_MM_A_NBJTX0_MMAX_A', 'NBJRR50_MM_A_NBJRR50_MMAX_A', 'NBJRR10_MM_A_NBJRR10_MMAX_A', 'NBJFXI3S28_MM_A_NBJFXI3S28_MMAX_A', 'IND_REGION', 'IND_SNV_REGION', 'LOG_REGION', 'TN_VOR_MM_A_TN_VOR_MMAX_A', 'NBJFF16_MM_A_NBJFF16_MMAX_A', 'NBJTN5_MM_A_NBJTN5_MMAX_A', 'RRAB_VOR_MM_A_RRAB_VOR_MMAX_A', 'NBJFF10_MM_A_NBJFF10_MMAX_A', 'NBJFXY8_MM_A_NBJFXY8_MMAX_A', 'NBJRR30_MM_A_NBJRR30_MMAX_A', 'SURFACE6', 'TAILLE1', 'TAILLE2', 'HAUTEUR_MAX', 'BDTOPO_BAT_MAX_HAUTEUR_MAX', 'HAUTEUR', 'BDTOPO_BAT_MAX_HAUTEUR', 'DISTANCE_213', 'PROPORTION_21', 'DISTANCE_411', 'CARACT4', 'DISTANCE_323', 'DISTANCE_212', 'DISTANCE_335', 'DISTANCE_334', 'KAPITAL32', '

Log Loss Train: 0.8622874280647036


Log Loss Dev:   0.9536605284485395


## Result

Compare `Log Loss Dev` above against the shipped arm's **0.91991** and the 2025
winner's **0.91936**. Every other setting is identical, so the difference is the
selector and nothing else.
